In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
import nltk
from nltk.tokenize import word_tokenize
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from collections import Counter
from tqdm import tqdm
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import precision_recall_fscore_support
import time
import os
import re
import nltk
import string
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
# from contractions import fix  # pip install contractions

nltk.download('punkt')
nltk.download('wordnet')
nltk.download('stopwords')

# Load spacy English model
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()
# nltk.data.path.append('/bohr/train-t05i/v2/punkt') #Import word library

# # Load the training set of the news dataset.
# train_df = pd.read_csv("/bohr/train-t05i/v2/train_news.csv")
nltk.data.path.append('./punkt') #Import word library

# Load the training set of the news dataset.
train_df = pd.read_csv("./train_news.csv")



def simple_tokenize(text):
    text = re.sub(r'@\w+', '', text)
    text = re.sub(r'http\S+|www\S+', '', text)

    text = re.sub(r'#(\w+)', r'\1', text)

    text = fix(text)

    text = text.lower()

    text = re.sub(r'[%s]' % re.escape(string.punctuation), '', text)
    text = re.sub(r'\d+', '', text)
    text = re.sub(r'(.)\1{2,}', r'\1\1', text)

    tokens = nltk.word_tokenize(text)

    tokens = [word for word in tokens if word not in stop_words]

    tokens = [lemmatizer.lemmatize(word) for word in tokens]
    return tokens



# When the number of words in a paragraph (including punctuation) exceeds 500, truncate the sentence.
# When the number of words is less than 500, pad the end of the sentence with a uniform number.
def preprocess(df, word2idx=None, label2idx=None, max_len=500, tokenize_func=None):
    texts = df["text"].values
    labels = df["category"].values if "category" in df.columns else None #Check if there is a label, if not, fill in "None".

    # Label encoding.
    if label2idx is None and labels is not None:
        unique_labels = set(labels)
        label2idx = {label: idx for idx, label in enumerate(unique_labels)}
    if labels is not None:
        labels = [label2idx[label] for label in labels]

    # Tokenization.
    tokenized_texts = [word_tokenize(text.lower()) for text in texts]

    if word2idx is None:
        # Build a vocabulary list.
        all_tokens = [token for text in tokenized_texts for token in text]
        vocab = Counter(all_tokens)
        vocab_size = 25000  #The total will not exceed 25,000 words.
        vocab = vocab.most_common(vocab_size - 2)
        word2idx = {word: idx + 2 for idx, (word, _) in enumerate(vocab)}
        word2idx["<unk>"] = 0
        word2idx["<pad>"] = 1

    # Numericalization
    def encode_text(text):
        return [word2idx.get(word, word2idx["<unk>"]) for word in text]

    encoded_texts = [encode_text(text) for text in tokenized_texts]

    # Fill or truncate.
    padded_texts = [
        (
            text[:max_len]
            if len(text) > max_len
            else text + [word2idx["<pad>"]] * (max_len - len(text))
        )
        for text in encoded_texts
    ]

    return padded_texts, labels, word2idx, label2idx


# Load training set.
X_train, y_train, word2idx, label2idx = preprocess(train_df,tokenize_func=simple_tokenize)


# Define dataset class
class NewsDataset(Dataset):
    def __init__(self, texts, labels):
        self.texts = torch.tensor(texts, dtype=torch.long)
        self.labels = torch.tensor(labels, dtype=torch.long) if labels is not None else None

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        return self.texts[idx], self.labels[idx] if self.labels is not None else self.texts[idx]

train_dataset = NewsDataset(X_train, y_train)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)


# Print the labels
label_name = label2idx
print(label_name)
num_labels = len(label_name)
print("The numbers of different label values can be：",num_labels)

flattened_array = [item for sublist in X_train for item in sublist] # Flatten a two-dimensional array into a one-dimensional array.
unique_elements = set(flattened_array) # Remove duplicate elements.
num_unique_elements = len(unique_elements)
print("The number of different words (including punctuation) in the training set’s features is:", num_unique_elements-1)

# Define a bi-directional LSTM
class LSTMClassifier(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, num_classes, pad_idx):
        super(LSTMClassifier, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim,padding_idx=pad_idx)
        self.lstm = nn.LSTM(embedding_dim, hidden_dim, batch_first=True, bidirectional = True) #
        self.fc = nn.Linear(2 * hidden_dim, num_classes) #
    def forward(self, x):
        embedded = self.embedding(x)
        lstm_output, (hidden, cell) = self.lstm(embedded)
        last_hidden = torch.cat((hidden[-2], hidden[-1]), dim=1) #
        logits = self.fc(last_hidden)
        return logits

# Model parameters
embedding_dim = 128
hidden_dim = 64
vocab_size = len(word2idx)
num_classes = 4
pad_idx = word2idx["<pad>"]

# Instantiate the model.
model = LSTMClassifier(vocab_size, embedding_dim, hidden_dim, num_classes, pad_idx)

# Define the loss function and optimizer.
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters())

# Train model
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)
model = model.to(device)
criterion = criterion.to(device)
num_epochs = 10
history = {"train_loss": [], "train_acc": []}
def categorical_accuracy(preds, y):
    max_preds = preds.argmax(dim=1, keepdim=True)
    correct = max_preds.squeeze(1).eq(y)
    return correct.sum() / y.shape[0]
for epoch in range(num_epochs):
    epoch_loss = 0
    epoch_acc = 0
    model.train()
    for batch in train_loader:
        optimizer.zero_grad()
        inputs, targets = batch
        inputs = inputs.to(device)
        targets = targets.to(device)
        logits = model(inputs)
        loss = criterion(logits, targets)
        acc = categorical_accuracy(logits, targets)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
        epoch_acc += acc.item()
    history["train_loss"].append(epoch_loss / len(train_loader))
    history["train_acc"].append(epoch_acc / len(train_loader))
    print(f'Epoch {epoch+1}/{num_epochs}, Xiao Ai Train Loss: {epoch_loss/len(train_loader):.4f}, Xiao Ai Train Accuracy: {epoch_acc/len(train_loader):.4f}')

[nltk_data] Downloading package punkt to /Users/tudor/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package wordnet to /Users/tudor/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package stopwords to /Users/tudor/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


{'tech': 0, 'sport': 1, 'business': 2, 'entertainment': 3}
The numbers of different label values can be： 4
The number of different words (including punctuation) in the training set’s features is: 21683
cpu
Epoch 1/10, Xiao Ai Train Loss: 1.3732, Xiao Ai Train Accuracy: 0.3018
Epoch 2/10, Xiao Ai Train Loss: 1.3226, Xiao Ai Train Accuracy: 0.4139
Epoch 3/10, Xiao Ai Train Loss: 1.2706, Xiao Ai Train Accuracy: 0.4961
Epoch 4/10, Xiao Ai Train Loss: 1.2091, Xiao Ai Train Accuracy: 0.5535
Epoch 5/10, Xiao Ai Train Loss: 1.1345, Xiao Ai Train Accuracy: 0.6145
Epoch 6/10, Xiao Ai Train Loss: 1.0418, Xiao Ai Train Accuracy: 0.6732
Epoch 7/10, Xiao Ai Train Loss: 0.9204, Xiao Ai Train Accuracy: 0.7408
Epoch 8/10, Xiao Ai Train Loss: 0.7578, Xiao Ai Train Accuracy: 0.7959
Epoch 9/10, Xiao Ai Train Loss: 0.5716, Xiao Ai Train Accuracy: 0.8703
Epoch 10/10, Xiao Ai Train Loss: 0.4192, Xiao Ai Train Accuracy: 0.9373


In [9]:
if os.environ.get('DATA_PATH'):
    data_path = os.environ.get("DATA_PATH") + "/"  
else:
    print("When the baseline is running, this error message will appear because the test set cannot be read, which is a normal phenomenon.") #When the baseline is running, this error message will appear because the test set cannot be read, which is a normal phenomenon.
test_df = pd.read_csv(data_path+"test_news_nolabel.csv")
# Use the text data from the test dataset to generate predicted category labels.
X_test, _, _, _ = preprocess(test_df, tokenize_func=simple_tokenize)
test_dataset = NewsDataset(X_test, None)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)
# Store the predicted categories.
predicted_labels = []

with torch.no_grad():
    for batch in test_loader:
        inputs, _ = batch
        inputs = inputs.to(device)
        logits = model(inputs)
        _, predicted = torch.max(logits, 1)
        predicted_labels.extend(predicted.cpu().numpy())

# Convert the predicted categories back to the same category format as the original dataset.
test_df["category"] = predicted_labels
test_df['category'] = test_df['category'].map(lambda x: list(label2idx.keys())[list(label2idx.values()).index(x)])

# Save the test dataset containing the predicted categories.
output_path = "submission.csv"
test_df.to_csv(output_path, index=False)
print("submission.csv is generated successfully")

When the baseline is running, this error message will appear because the test set cannot be read, which is a normal phenomenon.


NameError: name 'data_path' is not defined